# 05 · LangGraph Fundamentals: the Orchestration Engine

Notebook 04 built *one* agent. The research system needs many agents coordinated: a planner, N parallel
researchers, a fact-checker, a writer, a critic that can send work *back*, and a human who can approve the plan.

**LangGraph** models this as a **state machine / graph**:
- **State**: one typed dictionary shared by all nodes (the "blackboard" agents communicate through).
- **Nodes**: functions `state -> partial update`. An agent is just a node.
- **Edges**: what runs next: fixed, conditional (a router function), or dynamic fan-out (`Send`).
- **Checkpointer**: saves state after every step → resume, time travel, human-in-the-loop.

This notebook uses **plain Python functions instead of LLMs** so that every concept is isolated, deterministic and
free to run. Notebook 07 plugs the real agents into exactly these patterns.

| § | Concept | Used in the research system for |
|---|---|---|
| 1 | StateGraph, nodes, edges | the overall pipeline |
| 2 | Reducers | merging results from parallel researchers |
| 3 | Conditional edges + loops | the critic → re-research loop |
| 4 | `Send` fan-out (map-reduce) | one researcher per sub-question, in parallel |
| 5 | Subgraphs | each researcher's private working memory |
| 6 | Checkpointing & time travel | resume, debugging, history |
| 7 | `interrupt` (human-in-the-loop) | plan approval |
| 8 | Streaming | live progress in the UI |
| 9 | Retries & error handling | surviving failures |

In [1]:
import operator, time
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

## 1 · The smallest useful graph

A node receives the **current state** and returns **only the keys it wants to change**. LangGraph merges that partial
update into the state. Nodes never call each other: they communicate only through state, which keeps each agent
independent and testable.

In [2]:
class PipelineState(TypedDict):
    query: str
    plan: list[str]
    report: str


def planner(state: PipelineState) -> dict:
    return {"plan": [f"{state['query']}: background", f"{state['query']}: recent developments"]}


def writer(state: PipelineState) -> dict:
    return {"report": " | ".join(state["plan"])}


builder = StateGraph(PipelineState)
builder.add_node("planner", planner)
builder.add_node("writer", writer)
builder.add_edge(START, "planner")
builder.add_edge("planner", "writer")
builder.add_edge("writer", END)
graph = builder.compile()          # validates the structure and returns a runnable

graph.invoke({"query": "solid-state batteries"})

{'query': 'solid-state batteries',
 'plan': ['solid-state batteries: background',
  'solid-state batteries: recent developments'],
 'report': 'solid-state batteries: background | solid-state batteries: recent developments'}

## 2 · Reducers: how updates are merged

By default, a node's update **overwrites** the key. That breaks when *several nodes write the same key in the same
step*, which is exactly what parallel researchers do. LangGraph refuses ambiguous writes with `InvalidUpdateError`.

A **reducer** declares how to combine values: `Annotated[list, operator.add]` means "concatenate lists".
Let's see the failure first, then the fix.

In [3]:
from langgraph.errors import InvalidUpdateError


def build_parallel(state_cls):
    b = StateGraph(state_cls)
    b.add_node("researcher_a", lambda s: {"findings": ["A found X"]})
    b.add_node("researcher_b", lambda s: {"findings": ["B found Y"]})
    b.add_edge(START, "researcher_a")      # two edges from START = both run in the same step
    b.add_edge(START, "researcher_b")
    b.add_edge("researcher_a", END)
    b.add_edge("researcher_b", END)
    return b.compile()


class NoReducer(TypedDict):
    findings: list[str]

class WithReducer(TypedDict):
    findings: Annotated[list[str], operator.add]

try:
    build_parallel(NoReducer).invoke({"findings": []})
except InvalidUpdateError as e:
    print("Without reducer ->", type(e).__name__, ":", str(e).splitlines()[0])

print("With reducer    ->", build_parallel(WithReducer).invoke({"findings": []}))

Without reducer -> InvalidUpdateError : At key 'findings': Can receive only one value per step. Use an Annotated key to handle multiple values.
With reducer    -> {'findings': ['A found X', 'B found Y']}


## 3 · Conditional edges and loops (the critic loop)

`add_conditional_edges(node, router)` calls `router(state)` after the node; its return value names the next node.
This lets the graph **loop**: the critic can send the flow back to research.

Every loop needs a **termination guarantee**. We use an iteration counter in state *and* LangGraph's global
`recursion_limit` (max super-steps) as a safety net.

In [4]:
class LoopState(TypedDict):
    draft_quality: int
    iteration: int
    log: Annotated[list[str], operator.add]


MAX_ITERATIONS = 3

def research(state: LoopState) -> dict:
    return {"draft_quality": state["draft_quality"] + 30, "iteration": state["iteration"] + 1,
            "log": [f"research round {state['iteration'] + 1}"]}

def critic(state: LoopState) -> dict:
    return {"log": [f"critic: quality={state['draft_quality']}"]}

def route_after_critic(state: LoopState) -> str:
    if state["draft_quality"] >= 80:
        return "approve"
    if state["iteration"] >= MAX_ITERATIONS:
        return "give_up"                     # bounded: never loop forever
    return "revise"

b = StateGraph(LoopState)
b.add_node("research", research)
b.add_node("critic", critic)
b.add_edge(START, "research")
b.add_edge("research", "critic")
b.add_conditional_edges("critic", route_after_critic, {"revise": "research", "approve": END, "give_up": END})
loop_graph = b.compile()

for start_quality in (10, -100):
    out = loop_graph.invoke({"draft_quality": start_quality, "iteration": 0, "log": []})
    print(f"start={start_quality:>4}: iterations={out['iteration']} -> {out['log'][-1]}")

start=  10: iterations=3 -> critic: quality=100
start=-100: iterations=3 -> critic: quality=-10


In [5]:
# The graph structure as Mermaid source (renders on GitHub / mermaid.live); dotted lines are conditional edges.
print(loop_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	research(research)
	critic(critic)
	__end__([<p>__end__</p>]):::last
	__start__ --> research;
	critic -. &nbsp;approve&nbsp; .-> __end__;
	critic -. &nbsp;revise&nbsp; .-> research;
	research --> critic;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 4 · `Send`: dynamic fan-out / map-reduce

The number of researchers isn't known until the Planner runs (3–5 sub-questions). Static edges can't express that.
A router can instead return a **list of `Send(node, payload)`** objects: LangGraph launches one instance of `node`
per `Send`, **in parallel**, each receiving its own private payload (not the whole state).
Their outputs are merged with the reducer, and the next node runs once all of them finish (the "reduce" step).

We simulate researchers that each take 1 second. Sequential would take ~N seconds; parallel takes ~1.

In [6]:
from langgraph.types import Send


class ResearchState(TypedDict):
    sub_questions: list[str]
    findings: Annotated[list[str], operator.add]
    summary: str


class ResearcherInput(TypedDict):          # what each parallel researcher receives
    sub_question: str


def fan_out(state: ResearchState) -> list[Send]:
    return [Send("researcher", {"sub_question": q}) for q in state["sub_questions"]]

def researcher(payload: ResearcherInput) -> dict:
    time.sleep(1.0)                         # stands in for ~10 s of LLM + search work
    return {"findings": [f"finding about '{payload['sub_question']}'"]}

def aggregate(state: ResearchState) -> dict:
    return {"summary": f"{len(state['findings'])} findings collected"}

b = StateGraph(ResearchState)
b.add_node("researcher", researcher)
b.add_node("aggregate", aggregate)
b.add_conditional_edges(START, fan_out, ["researcher"])
b.add_edge("researcher", "aggregate")
b.add_edge("aggregate", END)
map_reduce = b.compile()

questions = ["costs", "safety", "adoption", "regulation", "competitors"]
t0 = time.perf_counter()
out = map_reduce.invoke({"sub_questions": questions, "findings": []})
print(f"{out['summary']} in {time.perf_counter() - t0:.2f}s (sequential would be ~{len(questions)}s)")
for f in out["findings"]:
    print(" -", f)

5 findings collected in 1.01s (sequential would be ~5s)
 - finding about 'costs'
 - finding about 'safety'
 - finding about 'adoption'
 - finding about 'regulation'
 - finding about 'competitors'


## 5 · Subgraphs: private working memory

A researcher's ReAct loop produces lots of intermediate messages (tool calls, raw search results). The main graph
should only receive the **final finding**, not that noise: otherwise the shared state (and the Writer's context)
balloons.

A **subgraph** is a compiled graph used as a node. It can have its *own* state schema. Here the subgraph keeps
`scratchpad` private and the wrapper node returns only `findings` to the parent. (In notebook 07 the subgraph is
the `create_agent` researcher from notebook 04.)

In [7]:
class ResearcherPrivateState(TypedDict):
    sub_question: str
    scratchpad: Annotated[list[str], operator.add]    # never leaves the subgraph
    finding: str


def search_step(s: ResearcherPrivateState) -> dict:
    return {"scratchpad": [f"raw results for {s['sub_question']}", "page text ...", "more noise ..."]}

def summarise_step(s: ResearcherPrivateState) -> dict:
    return {"finding": f"{s['sub_question']}: concise finding (from {len(s['scratchpad'])} raw items)"}

sb = StateGraph(ResearcherPrivateState)
sb.add_node("search", search_step)
sb.add_node("summarise", summarise_step)
sb.add_edge(START, "search")
sb.add_edge("search", "summarise")
sb.add_edge("summarise", END)
researcher_subgraph = sb.compile()


def run_researcher(payload: ResearcherInput) -> dict:
    """Adapter between parent and subgraph schemas: send the question in, bring only the finding out."""
    result = researcher_subgraph.invoke({"sub_question": payload["sub_question"], "scratchpad": []})
    return {"findings": [result["finding"]]}

b = StateGraph(ResearchState)
b.add_node("researcher", run_researcher)
b.add_node("aggregate", aggregate)
b.add_conditional_edges(START, fan_out, ["researcher"])
b.add_edge("researcher", "aggregate")
b.add_edge("aggregate", END)
out = b.compile().invoke({"sub_questions": ["costs", "safety"], "findings": []})
print("parent state keys:", list(out))          # no 'scratchpad' here
print(out["findings"])

parent state keys: ['sub_questions', 'findings', 'summary']
['costs: concise finding (from 3 raw items)', 'safety: concise finding (from 3 raw items)']


## 6 · Checkpointing and time travel

A **checkpointer** saves a snapshot of the state after every super-step, keyed by a `thread_id`. This gives us:
- **Resume**: a crashed/interrupted run continues from the last checkpoint instead of restarting (saving API calls).
- **History / time travel**: inspect every intermediate state; replay or fork from any point (debugging!).
- **Human-in-the-loop**: pausing (§7) is only possible because state is persisted.

`InMemorySaver` is for experiments; production uses `SqliteSaver` (same interface, persisted to a file).

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

# Re-build the critic-loop graph from §3, this time compiled WITH a checkpointer.
b = StateGraph(LoopState)
b.add_node("research", research)
b.add_node("critic", critic)
b.add_edge(START, "research")
b.add_edge("research", "critic")
b.add_conditional_edges("critic", route_after_critic, {"revise": "research", "approve": END, "give_up": END})
checkpointed = b.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "run-42"}}
checkpointed.invoke({"draft_quality": 10, "iteration": 0, "log": []}, config)

print("Current state:", checkpointed.get_state(config).values["log"][-1])
print("\nHistory (newest first):")
for snap in checkpointed.get_state_history(config):
    print(f"  step {snap.metadata['step']:>2} | next={snap.next} | quality={snap.values.get('draft_quality')}")

Current state: critic: quality=100

History (newest first):
  step  6 | next=() | quality=100
  step  5 | next=('critic',) | quality=100
  step  4 | next=('research',) | quality=70
  step  3 | next=('critic',) | quality=70
  step  2 | next=('research',) | quality=40
  step  1 | next=('critic',) | quality=40
  step  0 | next=('research',) | quality=10
  step -1 | next=('__start__',) | quality=None


## 7 · Human-in-the-loop with `interrupt`

Calling `interrupt(payload)` inside a node **pauses the graph** and returns `payload` to the caller (e.g. the UI shows
the research plan). The state is checkpointed. Later, invoking the graph with `Command(resume=value)` on the same
`thread_id` continues *inside that node*, where `interrupt(...)` now returns `value`.

This is how the user will approve or edit the research plan before any (expensive) research starts.

In [9]:
from langgraph.types import Command, interrupt


class ApprovalState(TypedDict):
    query: str
    plan: list[str]
    approved_plan: list[str]
    report: str


def propose_plan(state: ApprovalState) -> dict:
    return {"plan": ["background", "market size", "key players"]}

def human_review(state: ApprovalState) -> dict:
    decision = interrupt({"question": "Approve or edit this plan?", "plan": state["plan"]})
    return {"approved_plan": decision.get("edited_plan") or state["plan"]}

def research_and_write(state: ApprovalState) -> dict:
    return {"report": f"Report covering: {', '.join(state['approved_plan'])}"}

b = StateGraph(ApprovalState)
b.add_node("propose_plan", propose_plan)
b.add_node("human_review", human_review)
b.add_node("research_and_write", research_and_write)
b.add_edge(START, "propose_plan")
b.add_edge("propose_plan", "human_review")
b.add_edge("human_review", "research_and_write")
b.add_edge("research_and_write", END)
hitl = b.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "user-session-1"}}
paused = hitl.invoke({"query": "EV batteries"}, cfg)
print("Paused! Graph asks:", paused["__interrupt__"][0].value)
print("Next node waiting:", hitl.get_state(cfg).next)

Paused! Graph asks: {'question': 'Approve or edit this plan?', 'plan': ['background', 'market size', 'key players']}
Next node waiting: ('human_review',)


In [10]:
# ...the user edits the plan in the UI, then we resume the SAME thread:
final = hitl.invoke(Command(resume={"edited_plan": ["background", "key players", "regulation"]}), cfg)
print(final["report"])

Report covering: background, key players, regulation


## 8 · Streaming progress

Research runs take minutes; the UI must show what's happening. `graph.stream(...)` yields events while the graph runs:
- `stream_mode="updates"`: after each node, `{node_name: its_update}`.
- `stream_mode="custom"`: arbitrary events emitted from *inside* a node with `get_stream_writer()`
  (e.g. "researcher 2: searching arXiv...").
- `stream_mode="messages"`: LLM tokens as they are generated (for streaming the report text).
Several modes can be combined; each event is then a `(mode, payload)` tuple.

In [11]:
from langgraph.config import get_stream_writer


def researcher_with_progress(payload: ResearcherInput) -> dict:
    write = get_stream_writer()
    write({"status": f"searching for '{payload['sub_question']}'"})
    time.sleep(0.3)
    return {"findings": [f"finding: {payload['sub_question']}"]}

b = StateGraph(ResearchState)
b.add_node("researcher", researcher_with_progress)
b.add_node("aggregate", aggregate)
b.add_conditional_edges(START, fan_out, ["researcher"])
b.add_edge("researcher", "aggregate")
b.add_edge("aggregate", END)
streaming_graph = b.compile()

for mode, event in streaming_graph.stream({"sub_questions": ["costs", "safety"], "findings": []},
                                          stream_mode=["custom", "updates"]):
    print(f"{mode:>7}: {event}")

 custom: {'status': "searching for 'costs'"}
 custom: {'status': "searching for 'safety'"}


updates: {'researcher': {'findings': ['finding: costs']}}
updates: {'researcher': {'findings': ['finding: safety']}}
updates: {'aggregate': {'summary': '2 findings collected'}}


## 9 · Retries and error handling

Failures are normal in this system (rate limits, timeouts, flaky search). LangGraph offers two node-level tools:

1. **`retry_policy=RetryPolicy(...)`**: re-run a failing node with exponential backoff.
2. **`error_handler=fn`**: when a node still fails after its retries, `fn(state, error: NodeError)` runs instead
   and returns a `Command` (state update + where to go next). This turns a crash into *graceful degradation*.

### 9a · Retry policy on a flaky node

In [12]:
from langgraph.types import RetryPolicy

calls = {"n": 0}

def flaky_search(state: PipelineState) -> dict:
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError(f"attempt {calls['n']}: rate limited")
    return {"plan": [f"succeeded on attempt {calls['n']}"]}

b = StateGraph(PipelineState)
b.add_node("search", flaky_search,
           retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.1, backoff_factor=2.0,
                                    retry_on=ConnectionError))
b.add_edge(START, "search")
b.add_edge("search", END)
print(b.compile().invoke({"query": "x"})["plan"])

['succeeded on attempt 3']


### 9b · Error handler: recover instead of crash

Note two details we discovered while testing:
- The handler must return a **`Command`**.
- After recovery, the failed node's normal outgoing edges do **not** fire, so the handler must say where to go
  (`goto=`), otherwise the run simply ends.

In [13]:
from langgraph.errors import NodeError


class RobustState(TypedDict):
    query: str
    draft: str
    warnings: Annotated[list[str], operator.add]
    report: str


def fact_check(state: RobustState) -> dict:
    raise RuntimeError("fact-check model unavailable")

def on_fact_check_error(state: RobustState, error: NodeError) -> Command:
    # Degrade gracefully: skip fact-checking, record a warning, continue to the writer.
    return Command(update={"warnings": [f"{error.node} skipped: {error.error}"]}, goto="write")

def write(state: RobustState) -> dict:
    return {"report": f"Report on {state['query']} (warnings: {state['warnings']})"}

b = StateGraph(RobustState)
b.add_node("fact_check", fact_check, error_handler=on_fact_check_error)
b.add_node("write", write)
b.add_edge(START, "fact_check")
b.add_edge("fact_check", "write")
b.add_edge("write", END)
print(b.compile().invoke({"query": "EV batteries", "draft": "", "warnings": []})["report"])

Report on EV batteries (warnings: ['fact_check skipped: fact-check model unavailable'])


### 9c · ⚠️ Error handlers and `Send` tasks

Our researchers run as `Send` tasks (§4). While building this notebook we found that in this LangGraph version
(1.2.17) the node `error_handler` is **not applied to `Send`-dispatched tasks**: one failing researcher fails the
whole run. We verify it below, then show the pattern we use instead: **catch inside the node** and return a
*failed finding*, so the other researchers' work is kept and the report notes the gap.

In [14]:
class FanState(TypedDict):
    sub_questions: list[str]
    findings: Annotated[list[str], operator.add]
    failures: Annotated[list[str], operator.add]


def risky_researcher(payload: ResearcherInput) -> dict:
    if payload["sub_question"] == "regulation":
        raise ValueError("search API exploded")
    return {"findings": [f"finding: {payload['sub_question']}"]}

def researcher_handler(state: ResearcherInput, error: NodeError) -> Command:
    return Command(update={"failures": [str(error.error)]}, goto=END)


def build_fan(node_fn, **node_kwargs):
    b = StateGraph(FanState)
    b.add_node("researcher", node_fn, **node_kwargs)
    b.add_conditional_edges(START, lambda s: [Send("researcher", {"sub_question": q}) for q in s["sub_questions"]],
                            ["researcher"])
    b.add_edge("researcher", END)
    return b.compile()

inputs = {"sub_questions": ["costs", "regulation", "safety"], "findings": [], "failures": []}

# 1) error_handler on a Send target: does it catch the failure?
try:
    build_fan(risky_researcher, error_handler=researcher_handler).invoke(inputs)
    print("error_handler caught it")
except ValueError as e:
    print("error_handler did NOT apply to the Send task -> run failed:", e)


# 2) The pattern we use: catch inside the node, return a structured failure.
def safe_researcher(payload: ResearcherInput) -> dict:
    try:
        return risky_researcher(payload)
    except Exception as e:
        return {"failures": [f"{payload['sub_question']}: {type(e).__name__}: {e}"]}

out = build_fan(safe_researcher).invoke(inputs)
print("findings:", out["findings"])
print("failures:", out["failures"])

error_handler did NOT apply to the Send task -> run failed: search API exploded
findings: ['finding: costs', 'finding: safety']
failures: ['regulation: ValueError: search API exploded']


## 10 · Observations & Decisions

**Observations**
- State + partial updates + reducers make agent communication explicit; parallel writes *require* reducers
  (`InvalidUpdateError` otherwise).
- `Send` gives true parallel fan-out: 5 × 1 s tasks finished in ≈1 s.
- Subgraphs keep each researcher's noisy working memory out of the shared state.
- Checkpointing gives history/time travel for free and is a prerequisite for `interrupt` + `Command(resume=...)`.
- `stream_mode=["custom", "updates"]` provides exactly the progress events a UI needs.
- `RetryPolicy` handles transient failures; `error_handler` turns persistent failures into graceful degradation,
  but it must return a `Command` with `goto`, and **it does not cover `Send` tasks** in LangGraph 1.2.17.

**Decisions**
1. `ResearchState` (TypedDict) with `Annotated[list, operator.add]` reducers for `findings`, `failures`, `warnings`.
2. Researchers: `Send` fan-out, one per sub-question; each wraps the notebook-04 agent and returns only a `Finding`.
3. Researcher nodes **catch their own exceptions** and return a failure record (Send tasks bypass `error_handler`).
   Sequential nodes (fact-checker, writer, critic) use `RetryPolicy` + `error_handler` with explicit `goto`.
4. Critic loop: router with `max_iterations` (default 2) + `recursion_limit` safety net.
5. Plan approval via `interrupt` (toggle in config); `SqliteSaver` checkpointer in production.
6. Progress via `get_stream_writer()` custom events + `updates` mode → FastAPI SSE in Phase 4.